# Mosta multilayer communication (DeepRUOTv2)

This notebook mirrors the MOSTA multilayer pipeline: SDE simulation -> classifier prediction -> Sankey + 3D communication.

## 0) Environment
Use the `DeepRUOTv2` kernel (or conda env).

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""
device = "cpu"


In [ ]:
import os

project_root = '/lustre/home/2200012126/spatial_data/CytoBridge-ST-1104'
os.chdir(project_root)
print('CWD:', os.getcwd())

## 1) Configure paths and plotting options
Adjust if your data locations differ.

In [ ]:
config_path = 'config/mosta_config.yaml'
annotation_csv = 'data/mosta_four_time_with_annotation.csv'
output_dir = 'evaluation/mosta_communication_only_deepruotv2'

label_color_json = None
color_h5ad = '/lustre/home/2200012126/spatial_data/CytoBridge-ST-1104/data/mosta_after_pp_with_ae_and_center.h5ad'

# SDE / classifier settings
n_samples = 51365
sde_dt = 0.05
classifier_epochs = 1000
classifier_hidden = 128

# Split-SDE settings (for interpolation points in 3D)
split_sde_dt = 0.01
split_sigma = 0.03
split_growth_alpha = 0.5

# Interpolated unseen timepoints (example: [0.5, 1.5])
interp_time_points = [-1, 1.5, 2.5]

# Use real data for observed timepoints (True) or split-simulated points (False)
use_real_for_observed = True

# Plot controls
reverse_time_order = True
sankey_min_flow = None  # None means no filtering
fate_focus_label = 'reaEGC'  # None means no filtering
fate_focus_mode = 'source'  # 'either' | 'source' | 'target'
fate_min_flow = None  # None means no filtering
comm_edge_threshold = 5.0  # communication edges: draw if > threshold

# 3D background
background_color = 'white'
font_color = 'black'
# 3D communication edge color
comm_edge_color = 'black'

# Anchor mode for 3D edges/flows
anchor_mode = 'centroid'  # 'centroid' | 'nearest'
anchor_subsample = 1000  # None uses all cells
# Endpoint markers (PAGA-like)
highlight_endpoints = True
endpoint_size = 6
endpoint_opacity = 0.9

# 3D slices-only render
slices_only = False
show_time_axis = False
show_legend = False
show_title = False
fig_width = 1240
fig_height = 1754

# Export settings
export_svg = True
export_pdf = True
export_png = True
png_scale = 2

# 2D SVG snapshots (per timepoint)
snapshot_dir = os.path.join(output_dir, 'timepoint_svg')
snapshot_point_size = 2.5
snapshot_alpha = 0.9

# 2D mosaic panel
mosaic_cols = 4
mosaic_cell_size = 2.2
mosaic_show_title = True

## 2) Load config, data, and models

In [ ]:
import json
import numpy as np
import pandas as pd
import torch

from evaluation.arista_helpers import (
    load_config,
    load_arista_df,
    load_models,
    simulate_sde_points,
    simulate_sde_points_split,
    train_mlp_classifier,
    predict_labels_for_trajectories,
    save_interpolated_attention,
    analyze_attention_by_celltype,
    plot_sankey,
    plot_3d_spatial_sankey_style,
)

config = load_config(config_path)
df, _ = load_arista_df(config)

df_anno = pd.read_csv(annotation_csv)
if 'Annotation' not in df_anno.columns:
    raise ValueError(f'Annotation column missing in {annotation_csv}')
if len(df_anno) != len(df):
    raise ValueError('Annotation CSV length does not match data CSV')

df = df.copy()
df['Annotation'] = df_anno['Annotation'].astype(str).values

dim = int(config['data']['dim'])
device = 'cuda' if torch.cuda.is_available() else 'cpu'

f_net, score_net, exp_dir, device = load_models(
    config,
    exp_name=config['exp']['name'],
    device=device,
    model_tag='model_final',
    score_tag='score_model',
)

print('Device:', device)
print('Experiment dir:', exp_dir)

## 3) Train classifier (MLP) on annotated data
This predicts labels for SDE trajectories.

In [ ]:
feature_cols = ['samples'] + [f'x{i}' for i in range(1, dim + 1)]
model, label_encoder, acc = train_mlp_classifier(
    df,
    feature_cols=feature_cols,
    label_col='Annotation',
    hidden_size=classifier_hidden,
    epochs=classifier_epochs,
)
print('Classifier accuracy:', acc)

## 4) Simulate SDE trajectories
Add interpolated timepoints by listing them in `interp_time_points`.

Non-split SDE is used for Sankey/fate flow labels.
Split SDE is used for interpolation points in 3D + communication.

In [ ]:
time_points = sorted(df['samples'].unique().tolist())
interp_time_points = [t for t in interp_time_points if t not in time_points]
ts_points = sorted(set(time_points + interp_time_points))

# Non-split SDE (for Sankey/fate flow labels)
sde_points, sde_weights = simulate_sde_points(
    df=df,
    dim=dim,
    f_net=f_net,
    score_net=score_net,
    time_index=0,
    n_samples=n_samples,
    ts_points=ts_points,
    dt=sde_dt,
    sigma=0.0,
    include_score=False,
    device=device,
)

need_split = (len(interp_time_points) > 0) or (not use_real_for_observed)
if need_split:
    sde_points_split = simulate_sde_points_split(
        df=df,
        dim=dim,
        f_net=f_net,
        score_net=score_net,
        time_index=0,
        n_samples=n_samples,
        ts_points=ts_points,
        dt=split_sde_dt,
        sigma=split_sigma,
        growth_alpha=split_growth_alpha,
        device=device,
)
else:
    sde_points_split = None

print('SDE points:', len(sde_points), 'timepoints')
if sde_points_split is not None:
    print('Split SDE points:', len(sde_points_split), 'timepoints')

## 5) Predict labels for each SDE timepoint
`predicted_labels_list` drives Sankey and fate flow ribbons.
`predicted_labels_split` labels split-SDE points for 3D/communication.

In [ ]:
predicted_labels_list = predict_labels_for_trajectories(
    sde_points=sde_points,
    ts_points=ts_points,
    model=model,
    label_encoder=label_encoder,
    feature_dim=dim,
    device=device,
)

if sde_points_split is not None:
    predicted_labels_split = predict_labels_for_trajectories(
        sde_points=sde_points_split,
        ts_points=ts_points,
        model=model,
        label_encoder=label_encoder,
        feature_dim=dim,
        device=device,
)
else:
    predicted_labels_split = None

print('Predicted labels list length:', len(predicted_labels_list))

## 6) Build AnnData per timepoint (real + interpolated)

In [ ]:
import anndata as ad

feature_cols = [f'x{i}' for i in range(1, dim + 1)]
adata_dict = {}
time_keys = [str(t) for t in ts_points]

sde_map = {str(ts_points[i]): np.array(sde_points[i], dtype=np.float32) for i in range(len(ts_points))}
pred_map = {str(ts_points[i]): np.asarray(predicted_labels_list[i]).astype(str) for i in range(len(ts_points))}

if sde_points_split is not None:
    sde_map_split = {str(ts_points[i]): np.array(sde_points_split[i], dtype=np.float32) for i in range(len(ts_points))}
    pred_map_split = {str(ts_points[i]): np.asarray(predicted_labels_split[i]).astype(str) for i in range(len(ts_points))}
else:
    sde_map_split = None
    pred_map_split = None

for t in ts_points:
    key = str(t)
    if use_real_for_observed and t in time_points:
        subset = df[df['samples'] == t]
        X = subset[feature_cols].values.astype(np.float32)
        labels = subset['Annotation'].astype(str).values
    else:
        if sde_map_split is None:
            raise ValueError('Split SDE points missing; set interp_time_points or use_real_for_observed=True')
        X = sde_map_split[key]
        labels = pred_map_split[key]
    adata = ad.AnnData(X=X)
    adata.obs['Annotation'] = labels
    adata.obsm['spatial'] = X[:, :2]
    adata_dict[key] = adata

print('Built adata_dict with', len(adata_dict), 'timepoints')

## 7) Color mapping (labels -> colors)

In [ ]:
import matplotlib.pyplot as plt

def load_label_to_color(labels, label_color_json=None, color_h5ad=None, annotation_key="Annotation"):
    if label_color_json and os.path.exists(label_color_json):
        with open(label_color_json, 'r', encoding='utf-8') as f:
            return json.load(f)
    if color_h5ad and os.path.exists(color_h5ad):
        try:
            import scanpy as sc
            adata = sc.read_h5ad(color_h5ad)
            key = annotation_key if annotation_key in adata.obs else None
            if key is None and annotation_key.lower() in adata.obs:
                key = annotation_key.lower()
            if key:
                colors_key = f"{key}_colors"
                colors = adata.uns.get(colors_key)
                if colors is not None:
                    categories = (
                        adata.obs[key].cat.categories
                        if hasattr(adata.obs[key], 'cat')
                        else sorted(adata.obs[key].unique())
                    )
                    return dict(zip(categories, colors))
        except Exception as exc:
            print(f"Color map load failed from {color_h5ad}: {exc}")

    unique_labels = list(dict.fromkeys(labels))
    cmap = plt.get_cmap('tab20')
    label_to_color = {}
    for idx, lab in enumerate(unique_labels):
        rgb = cmap(idx % cmap.N)[:3]
        label_to_color[str(lab)] = '#{:02x}{:02x}{:02x}'.format(
            int(rgb[0] * 255), int(rgb[1] * 255), int(rgb[2] * 255)
        )
    return label_to_color

label_to_color = load_label_to_color(
    df['Annotation'].astype(str).values,
    label_color_json=label_color_json,
    color_h5ad=color_h5ad,
)

os.makedirs(output_dir, exist_ok=True)
with open(os.path.join(output_dir, 'label_to_color.json'), 'w', encoding='utf-8') as f:
    json.dump(label_to_color, f, indent=2)

print('Saved label_to_color.json')


## 8) 2D SVG snapshots per timepoint (nature-style, black background)

In [ ]:
import matplotlib.pyplot as plt

os.makedirs(snapshot_dir, exist_ok=True)

for tk in time_keys:
    ad = adata_dict[tk]
    coords = ad.obsm['spatial']
    labels = ad.obs['Annotation'].values
    colors = [label_to_color.get(l, '#888888') for l in labels]

    fig, ax = plt.subplots(figsize=(4.2, 4.2), dpi=300)
    fig.patch.set_facecolor(background_color)
    ax.set_facecolor(background_color)

    ax.scatter(
        coords[:, 0],
        coords[:, 1],
        s=snapshot_point_size,
        c=colors,
        linewidths=0,
        alpha=snapshot_alpha,
    )
    ax.set_aspect('equal')
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_title(f't = {tk}', color=font_color, fontsize=12, pad=6)

    out_path = os.path.join(snapshot_dir, f'time_{tk}.svg')
    fig.savefig(out_path, format='svg', facecolor=background_color, bbox_inches='tight')
    out_path = os.path.join(snapshot_dir, f'time_{tk}.pdf')
    fig.savefig(out_path, format='pdf', facecolor=background_color, bbox_inches='tight')
    plt.close(fig)

print('Saved SVGs to:', snapshot_dir)

## 9) 2D mosaic panel (small multiples)

In [ ]:
import math
import matplotlib.pyplot as plt

n_panels = len(time_keys)
cols = mosaic_cols
rows = math.ceil(n_panels / cols)
fig_w = cols * mosaic_cell_size
fig_h = rows * mosaic_cell_size

fig, axes = plt.subplots(rows, cols, figsize=(fig_w, fig_h), dpi=300)
fig.patch.set_facecolor(background_color)
axes = axes if isinstance(axes, np.ndarray) else np.array([[axes]])
axes = axes.reshape(rows, cols)

for idx, tk in enumerate(time_keys):
    r, c = divmod(idx, cols)
    ax = axes[r, c]
    ad = adata_dict[tk]
    coords = ad.obsm['spatial']
    labels = ad.obs['Annotation'].values
    colors = [label_to_color.get(l, '#888888') for l in labels]

    ax.set_facecolor(background_color)
    ax.scatter(
        coords[:, 0],
        coords[:, 1],
        s=snapshot_point_size,
        c=colors,
        linewidths=0,
        alpha=snapshot_alpha,
    )
    ax.set_aspect('equal')
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    if mosaic_show_title:
        ax.set_title(f'{tk}', color=font_color, fontsize=8, pad=3)

# Hide empty panels
for idx in range(n_panels, rows * cols):
    r, c = divmod(idx, cols)
    axes[r, c].axis('off')
    axes[r, c].set_facecolor(background_color)

mosaic_path = os.path.join(snapshot_dir, 'timepoint_mosaic.svg')
fig.savefig(mosaic_path, format='svg', facecolor=background_color, bbox_inches='tight')
plt.close(fig)

print('Saved mosaic:', mosaic_path)

## 10) Compute attention and communication per timepoint

In [ ]:
import pickle

attn_dir = os.path.join(output_dir, 'attention')
os.makedirs(attn_dir, exist_ok=True)

all_time_communications = {}
for t in ts_points:
    key = str(t)
    adata_t = adata_dict[key]
    print('Time', key, 'cells', adata_t.n_obs)

    attn_out = save_interpolated_attention(
        adata_t,
        time_value=float(t),
        f_net=f_net,
        device=device,
        out_dir=attn_dir,
    )

    comm = analyze_attention_by_celltype(
        edge_index=attn_out['edge_index'],
        attn=attn_out['attn_mean'],
        labels=adata_t.obs['Annotation'].values,
        spatial_coord=adata_t.obsm['spatial'],
        time_title=key,
        remove_self_loop=True,
        winsor_quantile=0.995,
        distance_bins=None,
        n_permutations=0,
    )
    all_time_communications[key] = comm

comm_path = os.path.join(output_dir, 'mosta_all_time_communications.pkl')
with open(comm_path, 'wb') as f:
    pickle.dump(all_time_communications, f)

print('Saved:', comm_path)

## 11) Sankey + 3D plots

In [ ]:
import pickle

# 读取保存的所有时间点通信数据
with open('evaluation/mosta_communication_only_deepruotv2/mosta_all_time_communications.pkl', 'rb') as f:
    all_time_communications = pickle.load(f)

In [ ]:
sankey_path = os.path.join(output_dir, 'lineage_sankey.html')
fig_sankey = plot_sankey(
    predicted_labels_list=predicted_labels_list,
    out_html=sankey_path,
    time_keys=time_keys,
    show_time_axis=True,
    min_flow=sankey_min_flow,
    label_to_color=label_to_color,
)

focus_source_only = fate_focus_mode == 'source'
focus_target_only = fate_focus_mode == 'target'

spatiotemporal_path = os.path.join(output_dir, 'spatiotemporal_3d.html')
fig_3d = plot_3d_spatial_sankey_style(
    adata_dict=adata_dict,
    all_time_communications=all_time_communications,
    time_keys=time_keys,
    label_to_color=label_to_color,
    predicted_labels_list=predicted_labels_list,
    spatial_key='spatial',
    z_spacing=3.0,
    reverse_time_order=reverse_time_order,
    intra_threshold=comm_edge_threshold,
    ribbon_min_count=fate_min_flow,
    ribbon_focus_celltype=fate_focus_label,
    ribbon_focus_source_only=focus_source_only if fate_focus_label else False,
    ribbon_focus_target_only=focus_target_only if fate_focus_label else False,
    background_color=background_color,
    font_color=font_color,
    anchor_mode=anchor_mode,
    anchor_subsample=anchor_subsample,
    highlight_endpoints=highlight_endpoints,
    endpoint_size=endpoint_size,
    endpoint_opacity=endpoint_opacity,
    edge_color=comm_edge_color,
    slices_only=slices_only,
    show_time_axis=show_time_axis,
    show_legend=show_legend,
    show_title=show_title,
    show_slice_border=True,
    observed_time_points=time_points,
    generated_time_points=interp_time_points,
    width=fig_width,
    height=fig_height,
    out_html=spatiotemporal_path,
)

print('Saved:', sankey_path)
print('Saved:', spatiotemporal_path)
fig_3d.update_layout(
    scene_camera=dict(
        eye=dict(x=1.7, y=1.0, z=0.9)
    ))
try:
    import plotly.io as pio
    if export_svg:
        pio.write_image(fig_sankey, os.path.join(output_dir, 'lineage_sankey.svg'))
        pio.write_image(fig_3d, os.path.join(output_dir, 'spatiotemporal_3d.svg'))
    if export_pdf:
        pio.write_image(fig_sankey, os.path.join(output_dir, 'lineage_sankey.pdf'))
        pio.write_image(fig_3d, os.path.join(output_dir, 'spatiotemporal_3d.pdf'))
    if export_png:
        pio.write_image(fig_3d, os.path.join(output_dir, 'spatiotemporal_3d.png'), width=fig_width, height=fig_height, scale=3)
    print('Exported vector/bitmap files.')
    print('Note: Plotly 3D exports are rasterized inside SVG/PDF.')
except Exception as e:
    print('Export failed:', e)


In [ ]:
import json
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

legend_path = os.path.join(snapshot_dir, "label_legend.svg")

handles = [
    Line2D([0], [0], marker='o', color='none',
           markerfacecolor=label_to_color[k], markersize=6, label=k)
    for k in label_to_color.keys()
]

fig, ax = plt.subplots(figsize=(4, 6), facecolor=background_color)
ax.set_facecolor(background_color)
ax.legend(handles=handles, loc='center left', frameon=False, labelcolor=font_color)
ax.axis('off')

fig.savefig(legend_path, format='svg', facecolor=background_color, bbox_inches='tight')
plt.close(fig)

print("Saved legend:", legend_path)
